Generating templates from Pre-processed maps

In [ ]:
import os
import numpy as np
import healpy as hp

import matplotlib.pyplot as plt
import matplotlib.path as mpath
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from cmcrameri import cm 
from matplotlib.patches import Ellipse
from PIL import Image

from scipy.odr import ODR, Model, Data
from scipy import constants
from scipy.stats import spearmanr
from scipy.ndimage import gaussian_filter

from astropy.io import fits
from astropy.wcs import WCS
from astropy.convolution import Gaussian2DKernel, convolve

from reproject import reproject_interp
from reproject.mosaicking import find_optimal_celestial_wcs, reproject_and_coadd


In [ ]:
LINUX_DIRECTORY = "/scratch/nas_comap1/jgalla/MPHYS_PROJECT"

SNR_CSV_PATH = f'{LINUX_DIRECTORY}/data/SNR_locations_full_final.csv'
PS_CSV_PATH = f"{LINUX_DIRECTORY}/data/Effelsberg_2.73GHz_(5'_smoothed)_point_sources_final.csv"

FWHM_PS = 5
THRESHOLD_FACTOR = 0.8

In [ ]:
# Main functions

def chisquare(residuals, errors):
    residuals = np.array(residuals)
    errors = np.array(errors)
    return np.sum(residuals**2/errors**2)


def cut_data(data, noise, wcs, header):
    
    lat_min = B_BOUNDS[0]
    lat_max = B_BOUNDS[1]
    long_min = L_BOUNDS[0]
    long_max = L_BOUNDS[1]
    step = header['CDELT2']  # pixel resolution

    # may be off by fraction of pixel
    start_row = int(header['CRPIX2'] + lat_min / step)
    end_row   = int(header['CRPIX2'] + lat_max / step)
    start_col = int(header['CRPIX1'] + (header['CRVAL1'] - long_max) / step)
    end_col   = int(header['CRPIX1'] + (header['CRVAL1'] - long_min) / step)
    
    # Safety check against invalid indices
    nrows, ncols = data.shape
    start_row = max(0, min(nrows - 1, start_row))
    end_row   = max(0, min(nrows - 1, end_row))
    start_col = max(0, min(ncols - 1, start_col))
    end_col   = max(0, min(ncols - 1, end_col))
    
    # Extract and process submap
    submap = data[start_row:end_row + 1, start_col:end_col + 1].copy()
    submap[submap == 0] = np.nan
    noise_submap = noise[start_row:end_row + 1, start_col:end_col + 1].copy()
    noise_submap[noise_submap == 0] = np.nan

    wcs.wcs.crpix[0] -= start_col
    wcs.wcs.crpix[1] -= start_row

    header['CRPIX1'] = wcs.wcs.crpix[0]
    header['CRPIX2'] = wcs.wcs.crpix[1]
    
    return submap, noise_submap, wcs, header

def rebin_array(data, factor):
    n_rows, n_cols = data.shape
    # print(f"Original shape: {data.shape}")

    # Crop rows and columns if not divisible by factor
    if n_rows % factor != 0:
        new_n_rows = (n_rows // factor) * factor
        data = data[:new_n_rows, :]
        # print(f"Adjusted rows to: {new_n_rows}")
    if n_cols % factor != 0:
        new_n_cols = (n_cols // factor) * factor
        data = data[:, :new_n_cols]
        # print(f"Adjusted columns to: {new_n_cols}")

    # Reshape and average in blocks of factor x factor
    reshaped_data = data.reshape(data.shape[0] // factor, factor, data.shape[1] // factor, factor)
    rebinned_data = reshaped_data.mean(axis=(1, 3))  # If any pixel in block is nan, whole block is nan (good)
    # print(f"Re-shaped data to: {rebinned_data.shape}")
    
    return rebinned_data


def rebin_map(data, header, factor):
    
    # l_min = L_BOUNDS[0]
    # l_max = L_BOUNDS[1]
    # b_min = B_BOUNDS[0]
    # b_max = B_BOUNDS[1]

    data = rebin_array(data, factor)

    # ny, nx = data.shape
    # cy, cx = ny // 2, nx // 2
    # print(cy, cx)   

    new_header = header.copy()
    # new_header["CRPIX1"] = 1
    # new_header["CRPIX2"] = 1
    # new_header["CRVAL1"] = l_max
    # new_header["CRVAL2"] = b_min
    new_header["CDELT1"] *= factor
    new_header["CDELT2"] *= factor
    # new_header["LONPOLE"] = 180
    
    # for k, v in new_header.items():
    #     print(f"{k}: {v}")
    new_wcs = WCS(new_header)

    return data, new_wcs, new_header


# TT plots
def tt_scatter_odr(datax, datay, datax_unc, datay_unc, x_freq, y_freq):
    # Initial intercept guess
    try:
        c_initial = min(datay)
    except Exception as E:
        c_initial = 0.0
        print(f"Warning: {E}\nForced initial intercept value to 0.0")
    
    # # estimate gradient given expected spectral index of -2.1 (requires frequencies)
    # m_initial, _ = beta_to_gradient(-2.1, 0, x_freq, y_freq) 
    
    # OR: estimate gradient by diff_y/diff_x
    try:
        m_initial1 = (max(datay) - min(datay)) / (datax[np.argmax(datay)] - datax[np.argmin(datay)])
        m_initial2 = (datay[np.argmax(datax)] - datay[np.argmin(datax)]) / (max(datax) - min(datax))
        m_initial = (m_initial1 + m_initial2)/2
    except Exception as E:
        m_initial, _ = beta_to_gradient(-2.1, 0, x_freq, y_freq)
        print(f"Warning: {E}\n Forced initial gradient value to be {m_initial}")
    
    # ODR
    linreg = Model(linreg_model_odr)
    data = Data(datax, datay, we=(1/(datax_unc**2)), wd=(1/(datay_unc**2)))
    odrobj = ODR(data, linreg, beta0=(m_initial, c_initial))
    odrout = odrobj.run()
    odrbeta = odrout.beta
    odrsdbeta = odrout.sd_beta

    return [odrbeta[0], odrsdbeta[0]], [odrbeta[1], odrsdbeta[1]]


def gradient_to_beta(gradient, gradient_err, freq1, freq2):
    factor = 1/np.log(freq1 / freq2)
    # If gradient <= 0 (log(x <= 0) --> RuntimeError), assign NaNs but mute error message to not clog terminal
    with np.errstate(invalid='ignore', divide='ignore'):
        beta = np.log(gradient)*factor
        beta_err = np.abs(gradient_err /gradient * factor)
    beta = np.where(np.isfinite(beta), beta, np.nan)
    beta_err = np.where(np.isfinite(beta_err), beta_err, np.nan)
    return beta, beta_err


def beta_to_gradient(beta, beta_err, freq1, freq2):
    
    factor = - np.log(freq1 / freq2) 
    gradient = np.exp(beta * factor)
    gradient_err = gradient * np.abs(beta_err) * factor
    
    return gradient, gradient_err


def full_TT_plot(x, y, x_model, y_model, xlabel='x', ylabel='y', title='title'):
    
    plt.scatter(x, y, label='data')
    plt.plot(x_model, y_model, label='Linear regression model', color='r')
    
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid()
    plt.legend(fontsize='small')
    plt.show()


# The following are for masking
def read_snr_catalogue_simple(file_path, include_uncertain=True):
    # From D. A. Green SNR Catalogue: https://www.mrao.cam.ac.uk/surveys/snrs/snrs.data.html

    x_coords = [] # galactic coords - l
    y_coords = [] # galactic coords - b
    diameters = [] # arcmin (diameter, not radius - see documentation on website)

    with open(file_path, 'r') as file:
        for i, line in enumerate(file):
            if i > 2: # skip headers
                valid = True
                arr = line.split()
            
                x_coord = float(arr[0])

                y_coord = float(arr[1])
                
                diameter_str = arr[7] 
                if '?' in diameter_str: 
                    diameter = split_diameter(diameter_str.replace('?', '')) 
                else:
                    diameter = split_diameter(diameter_str)
        
                x_coords.append(x_coord)
                y_coords.append(y_coord)
                diameters.append(diameter)
    
    return x_coords, y_coords, diameters


def find_point_sources(fits_data, fwhm=FWHM_PS, brightness_thres=THRESHOLD_FACTOR):
    
    fits_data = np.where(fits_data == hp.UNSEEN, np.nan, fits_data) # DAOStarFinder doesn't like hp.UNSEEN

    # Calculate std of whole image
    std = np.nanstd(fits_data) # nanstd ignores nans
    # print(f'STANDARD DEVIAION  = {std}')

    daofind =  DAOStarFinder(fwhm=fwhm, threshold = brightness_thres*std) # TWEAK
    # daofind =  DAOStarFinder(fwhm=fwhm, threshold = brightness_thres) # TWEAK
    sources = daofind(fits_data)

    if sources:
        print(f'\nNumber of point sources identified: {len(sources)}')
        # print(sources)
        x_coords = sources['xcentroid']
        y_coords = sources['ycentroid']
    else:
        print('No point sources identified')
        x_coords = []
        y_coords = []

    return x_coords, y_coords


def write_to_csv(x_coords, y_coords, radius=FWHM_PS/2, file_name='data.csv'):
    
    file_path = f'{CSV_SAVEDIR}/{file_name}'
    header = "l [deg], b [deg], Radius [arcmin]"

    radius_arr = np.full_like(x_coords, radius)
    data_arr = np.array([x_coords, y_coords, radius_arr]).T

    np.savetxt(file_path, data_arr, delimiter = ",", header=header)

    print(f'File saved as {file_path}')


def get_point_sources(file_path):
    
    data = np.genfromtxt(file_path, delimiter=',', skip_header=1, dtype=float)
    l_arr = data[:,0]
    b_arr = data[:,1]
    radius_arr = data[:,2]

    return l_arr, b_arr, radius_arr 


def create_point_source_mask(fits_data, x_coords, y_coords, fwhm):

    # Calculate sigma from FWHM
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))

    # Create meshgrid corresponding to image coords
    y, x = np.indices(fits_data.shape)

    # Create mask
    mask = np.zeros(fits_data.shape, dtype=bool) # empty, same shape as GDIGS mosaic
    for x_c, y_c in zip(x_coords, y_coords):
        distance = np.sqrt((x - x_c)**2 + (y - y_c)**2) 
        mask = np.logical_or(mask, distance < 3*sigma)  # if pixel is < 3*sigma away from centre of point source (i.e. inside Gaussian), mask it
    
    return mask


def find_SNRs(file_path = SNR_CSV_PATH):

    data = np.genfromtxt(file_path, delimiter=',', skip_header=1, dtype=float)
    l = data[:,0]
    b = data[:,1]
    x_radius = data[:,2] / 2
    y_radius = data[:,3] / 2
    
    return l, b, x_radius, y_radius


def create_snr_mask(fits_data, x_coords, y_coords, x_radii, y_radii):

    # Create meshgrid corresponding to image coords
    y, x = np.indices(fits_data.shape)

    # Create mask
    mask = np.zeros(fits_data.shape, dtype=bool) # empty, same shape as GDIGS mosaic
    for x_c, y_c, x_rad, y_rad in zip(x_coords, y_coords, x_radii, y_radii):

        ellipse_condition = ((x - x_c) ** 2) / (x_rad ** 2) + ((y - y_c) ** 2) / (y_rad ** 2) <= 1
        mask = np.logical_or(mask, ellipse_condition) # if pixel within SNR ellipse, mask it

    return mask


def create_mask(data, wcs, map_title, show_plots=False, show_masks=False):

    # Locate point sources
    l_ps, b_ps, radii_ps = find_point_sources(file_path=PS_CSV_PATH)
    x_ps, y_ps = galactic_to_pixel(l_ps, b_ps, WCS=wcs)
    if show_plots:
        plot_fits(data, wcs, x_coords=x_ps, y_coords=y_ps, title=f'{map_title} - Point Sources Locations')
    
    # Mask point sources
    mask_ps = create_point_source_mask(data, x_ps, y_ps, fwhm=radii_ps[0])
    if show_masks:
        plt.imshow(mask_ps, origin='lower', cmap='gray')
        plt.title('Point Source Mask')
        plt.show()
    if show_plots:
        data_ps = data.copy()
        data_ps[mask_ps] = np.nan # change to hp.UNSEEN?
        plot_fits(data_ps, wcs, title=f'{map_title} - Point Sources Masked')

    # Mask supernova remnants
    x_snr, y_snr, x_radii, y_radii = find_SNRs(file_path=SNR_CSV_PATH)
    x_snr, y_snr = galactic_to_pixel(x_snr, y_snr, wcs)

    if show_plots:
        plot_fits(data, wcs, x_coords=x_snr, y_coords=y_snr, title=f'{map_title} - SNR Locations')
    mask_snr = create_snr_mask(data, x_snr, y_snr, x_radii, y_radii)
    if show_masks:
        plt.imshow(mask_snr, origin='lower', cmap='gray')
        plt.title('SNR Mask')
        plt.show()
    if show_plots:
        data_snr = data.copy()
        data_snr[mask_snr] = np.nan # change to hp.UNSEEN?
        plot_fits(data_snr, wcs, title=f'{map_title} - SNRs Masked')

    # Combine masks
    mask_ps_snr = mask_ps | mask_snr
    if show_masks:
        plt.imshow(mask_ps_snr, origin='lower', cmap='gray')
        plt.title('SNR and PS Mask')
        plt.show()
    if show_plots:
        data_ps_snr = data.copy()
        data_ps_snr[mask_ps_snr] = np.nan # change to hp.UNSEEN?
        plot_fits(data_ps_snr, wcs, title=f'{map_title} - SNRs and Point Sources Masked')

    return mask_ps_snr

In [ ]:
# Main code for generating templates

